In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!apt-get update
!apt-get install -y ffmpeg

## CPU optimized video processor

In [ ]:
import os
import random
import subprocess
from pathlib import Path

# ── Kaggle Paths ─────────────────────────────────────────────────────
# CHANGE 'your-dataset-name' to the actual folder name in Kaggle's input pane!
INPUT_DIR = Path("/kaggle/input/datasets/pranay22077/dfdc-10/dfdc_train_part_06/dfdc_train_part_6") 
OUTPUT_DIR = Path("/kaggle/working/degraded_videos_06")

def main():
    print("☁️ Starting Cloud Degradation Pipeline on Kaggle...")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Grab all mp4s from the read-only input directory
    video_files = [f for f in INPUT_DIR.glob("**/*.mp4")]
    total = len(video_files)
    
    if total == 0:
        print("❌ No MP4 files found. Check your INPUT_DIR path!")
        return

    print(f"📦 Found {total} videos. Applying randomized compression...\n")

    for idx, video_path in enumerate(video_files, start=1):
        output_path = OUTPUT_DIR / f"{video_path.name}"
        
        if output_path.exists():
            continue

        # WhatsApp-style heavy compression (CRF 28-38)
        random_crf = random.randint(28, 38)
        
        # Scammer resolution downscaling
        resolutions = ["480", "360"]
        random_res = random.choice(resolutions)
        scale_filter = f"scale=-2:{random_res}"

        if idx % 50 == 0 or idx == 1:
            print(f"[{idx}/{total}] Processing: {video_path.name} (CRF: {random_crf}, Res: {random_res}p)")

        # FFmpeg command adapted for Linux cloud (no hardware acceleration flags needed)
        cmd = [
            "ffmpeg", 
            "-y", 
            "-i", str(video_path),
            "-vcodec", "libx264",
            "-preset", "ultrafast",   # CRITICAL FOR CLOUD: Speeds up CPU encoding drastically
            "-crf", str(random_crf),
            "-vf", scale_filter,
            "-acodec", "copy",
            "-loglevel", "error",
            str(output_path)
        ]

        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError:
            print(f"   ❌ FFmpeg failed to process {video_path.name}")
            if output_path.exists():
                output_path.unlink()

    print("\n✅ Cloud degradation complete. Files saved to /kaggle/working/")

if __name__ == "__main__":
    main()

In [ ]:
!zip -q -r /kaggle/working/degraded_videos_06.zip /kaggle/working/degraded_videos_06

In [ ]:
!ls /kaggle/input/datasets/pranay22077/dfdc-10/dfdc_train_part_05/dfdc_train_part_5/ | grep ".json"

In [ ]:
import shutil
from IPython.display import FileLink, display

# 1. Define where the file is, and where we want to put it
source_path = "/kaggle/input/datasets/pranay22077/dfdc-10/dfdc_train_part_02/dfdc_train_part_2/metadata.json"
# working_path = "/kaggle/working/degraded_videos_06/metadata.json"
working_path = "/kaggle/working/"
# 2. Copy it from the read-only input folder to your writable working folder
shutil.copy(source_path, working_path)
print("✅ File copied successfully!")

# 3. Generate a magical download link
print("⬇️ Click the link below to download your answer key:")
display(FileLink("metadata.json"))